# client-class-many-questions: The same client as a class

**`client-class-many-questions.py` does what `client-functions-one-question.py` does, but as a class, `MCPOpenAIClient`, that connects once and keeps the connection open, so it can answer many questions.** That's how a real host is built: A chat app connects to its servers at startup and answers questions as they arrive, over the same connection.

Read `client-functions-one-question.ipynb` first. This notebook covers only what's different.

## 1. Why `async with` isn't enough here

`async with` closes the connection when its block ends. In a class, you connect in one method (`connect_to_server`), use the connection in another (`process_query`), and close it in a third (`cleanup`). No single block surrounds all three.

`AsyncExitStack`, from Python's built-in `contextlib` module, solves this. It's a list of open things to close later:

- **`enter_async_context(x)`:** Opens `x`, the way `async with x` would, and remembers to close it.
- **`enter_context(x)`:** The same for an ordinary `with` thing, like an open file.
- **`aclose()`:** Closes everything it remembered, last opened first.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 connect_to_server  opens the log file, the server and the session
 process_query      uses self.session, which is still open
 cleanup            aclose() closes the session, server and file
</pre>

## 2. The class

`get_mcp_tools` and `process_query` are `client-functions-one-question`'s functions, with `self.` in front of the session, the OpenAI client and the model. The new parts are `__init__`, `connect_to_server` and `cleanup`.

In [1]:
import json
from contextlib import AsyncExitStack

from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI

load_dotenv()

True

In [2]:
class MCPOpenAIClient:
    def __init__(self, model="gpt-4o-mini"):
        self.model = model
        self.openai_client = AsyncOpenAI()
        self.exit_stack = AsyncExitStack()
        self.session = None

    async def connect_to_server(self, server_script_path="server.py"):
        server = StdioServerParameters(command="python", args=[server_script_path])
        log = self.exit_stack.enter_context(open("server_log.txt", "w"))

        read, write = await self.exit_stack.enter_async_context(stdio_client(server, errlog=log))
        self.session = await self.exit_stack.enter_async_context(ClientSession(read, write))
        await self.session.initialize()

    async def get_mcp_tools(self):
        tools_result = await self.session.list_tools()

        tools = []
        for tool in tools_result.tools:
            tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema,
                    },
                }
            )

        return tools

    async def process_query(self, query):
        tools = await self.get_mcp_tools()
        messages = [{"role": "user", "content": query}]

        response = await self.openai_client.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=tools,
        )
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        if assistant_message.tool_calls:
            for tool_call in assistant_message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                print("The model called:", name, args)

                result = await self.session.call_tool(name, args)
                messages.append(
                    {"role": "tool", "tool_call_id": tool_call.id, "content": result.content[0].text}
                )

            final_response = await self.openai_client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tools,
                tool_choice="none",
            )
            return final_response.choices[0].message.content
        else:
            return assistant_message.content

    async def cleanup(self):
        await self.exit_stack.aclose()

The `True` under the imports is `load_dotenv()` saying it found and loaded the `.env` file.

- **`__init__`:** Stores the model, the OpenAI client and an empty `AsyncExitStack`. `self.session` stays `None` until the client connects.
- **`connect_to_server`:** Opens the log file, starts the server and opens the session, each through the stack, so all three stay open after the method returns.
- **`cleanup`:** One `aclose()` closes all three.

## 3. Several questions on one connection

In [3]:
client = MCPOpenAIClient()
await client.connect_to_server()

questions = [
    "What is our company's vacation policy?",
    "How many days a week can I work from home?",
    "What is the capital of France?",
]
for question in questions:
    print(question)
    answer = await client.process_query(question)
    print(answer)
    print()

await client.cleanup()

What is our company's vacation policy?


The model called: get_knowledge_base {}


Our company's vacation policy states that full-time employees are entitled to 20 paid vacation days per year. Vacation days can be taken after completing 6 months of employment. Unused vacation days can be carried over to the next year up to a maximum of 5 days. To request vacation, employees must submit their requests at least 2 weeks in advance through the HR portal.

How many days a week can I work from home?


The model called: get_knowledge_base {}


You can work from home up to 3 days per week. Remote work days must be coordinated with your team and approved by your direct manager.

What is the capital of France?


The capital of France is Paris.



The first two questions needed company information, so the model called the tool each time. The third didn't, so the model answered straight away, without a tool call: That's the `else` branch of `process_query`. The model decides on every question, using the tool's description.

## 4. What the server logged

In [4]:
with open("server_log.txt") as log:
    print(log.read())

Processing request of type ListToolsRequest
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
Processing request of type CallToolRequest
Processing request of type ListToolsRequest



One server process handled all three questions: One `tools/list` per question, and one `tools/call` for each time the model used the tool.

## 5. The .py version

`client-class-many-questions.py` has the same class. Its `main()` asks the same three questions:

<div style="font-size: 0.85em">

```python
async def main():
    client = MCPOpenAIClient()
    await client.connect_to_server()

    questions = [
        "What is our company's vacation policy?",
        "How many days a week can I work from home?",
        "What is the capital of France?",
    ]
    for question in questions:
        print(question)
        answer = await client.process_query(question)
        print(answer)
        print()

    await client.cleanup()


asyncio.run(main())
```

</div>

Run from this folder with `python client-class-many-questions.py`, it printed:

<div style="font-size: 0.85em">

```text
What is our company's vacation policy?
The model called: get_knowledge_base {}
Our company's vacation policy states that full-time employees are entitled to 20 paid vacation days per year. Employees may take vacation days after completing 6 months of employment. Unused vacation days can be carried over to the next year, but the maximum carryover limit is 5 days. Vacation requests should be submitted at least 2 weeks in advance through the HR portal.

How many days a week can I work from home?
The model called: get_knowledge_base {}
You can work from home up to 3 days per week, provided that your remote work days are coordinated with your team and approved by your direct manager.

What is the capital of France?
The capital of France is Paris.
```

</div>

## Say it out loud

> "In a real host, I'd wrap the client in a class that connects once at startup and keeps the session open. Because connecting, using and closing happen in different methods, I can't use one `async with` block, so I use `AsyncExitStack`: It enters the stdio transport and the session, keeps them open, and closes them all in `cleanup`. Then each question reuses the same connection. The model still decides per question whether to call the tool, and for a general question like the capital of France, it answers directly."